In [1]:
from __future__ import annotations

# ---------------------------------------------------------------------
# IMPORTANT: cap BLAS thread count BEFORE numpy / sklearn are imported.
# When the GA training loop runs through joblib's loky backend (one
# process per seed), each worker still gets its own BLAS thread pool;
# without this cap, BLAS would oversubscribe cores and tank the speedup.
# os.environ.setdefault preserves any externally-set override.
# ---------------------------------------------------------------------
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS",      "1")
os.environ.setdefault("OMP_NUM_THREADS",      "1")
os.environ.setdefault("BLIS_NUM_THREADS",     "1")

import numpy
import random
import pandas
import time
from typing import Any

from joblib import Parallel, delayed

from deap import creator, base
# Define the DEAP creator classes in THIS (parent) process so worker
# results -- which contain creator.Individual / creator.IndividualSingle
# instances -- can be unpickled here. Each worker also recreates them
# lazily inside MultiObjectiveTraining.run / SingleObjectiveTraining.run
# (those guards are idempotent: `if name not in creator.__dict__`).
if "FitnessMulti" not in creator.__dict__:
    creator.create("FitnessMulti", base.Fitness, weights=(1.0, 1.0))
if "Individual" not in creator.__dict__:
    creator.create("Individual", list, fitness=creator.FitnessMulti)
if "FitnessSingle" not in creator.__dict__:
    creator.create("FitnessSingle", base.Fitness, weights=(1.0,))
if "IndividualSingle" not in creator.__dict__:
    creator.create("IndividualSingle", list, fitness=creator.FitnessSingle)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import matthews_corrcoef
from scipy.stats import pointbiserialr

from training_config import TrainingConfig
from training_utils import select_pareto_individual
from multi_objective_training import MultiObjectiveTraining
from single_objective_training import SingleObjectiveTraining
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression as _SFSLogReg
from evaluation_utils import (
    get_continuous_columns, get_dummy_columns,
    build_model_package, evaluate_and_save,
    compute_stability_matrix, plot_stability_heatmap, plot_feature_frequency,
    compute_vif_across_seeds, plot_vif_boxplot,
)
from evaluation_utils import apply_proportional_noise, apply_dummy_noise, evaluate_model
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression as _CmpLogReg
import matplotlib.pyplot as plt

In [2]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    numpy.random.seed(seed)

In [3]:
#CSV_TRAIN_PATH: str = "readmit/readmit_130_hospitals_preprocessed_train_data.csv"
#CSV_TEST_PATH: str = "readmit/readmit_130_hospitals_preprocessed_test_data.csv"
#TARGET_COLUMN: str = "target_readmitted"
CSV_TRAIN_PATH: str = "arrhythmia/arrhythmia_preprocessed_train_data.csv"
CSV_TEST_PATH: str = "arrhythmia/arrhythmia_preprocessed_test_data.csv"
TARGET_COLUMN: str = "Outcome"
#CSV_TRAIN_PATH: str = "hmeq/hmeq_preprocessed_train_data.csv"
#CSV_TEST_PATH: str = "hmeq/hmeq_preprocessed_test_data.csv"
#TARGET_COLUMN: str = "BAD"
#CSV_TRAIN_PATH: str = "D:\\RoutePatch\\GitWorking\\multi-objective-regression\\test_data\\rad_fusion_train_modified.csv"
#CSV_VALIDATION_PATH: str = "D:\\RoutePatch\\GitWorking\\multi-objective-regression\\test_data\\rad_fusion_validation_modified.csv"
#CSV_TEST_PATH: str = "D:\\RoutePatch\\GitWorking\\multi-objective-regression\\test_data\\rad_fusion_test_modified.csv"
#TARGET_COLUMN: str = "label"
#CSV_TRAIN_PATH: str = "synth_train_data.csv"
#CSV_TEST_PATH: str = "synth_test_data.csv"
#TARGET_COLUMN: str = "target"

RESULT_PATH: str = time.strftime("%Y-%m-%d_%H-%M-%S")
RESULT_PATH_MULTI: str = f"{RESULT_PATH}\\multi"
RESULT_PATH_SINGLE: str = f"{RESULT_PATH}\\single"
RESULT_PATH_EVAL: str = f"{RESULT_PATH}\\evaluation"

USE_KNEE_POINT_SELECTION: bool = False

# Parallelism for the GA training loop (cell 6).
#   -1 = use all available CPU cores (one worker process per seed, up to that many in flight).
#    1 = pure sequential loop (good for debugging / profiling / when each seed is already huge).
# Any other positive int caps the number of concurrent workers.
# Numerical results are independent of N_JOBS by construction: each seed
# is self-contained (set_seed -> MOGA -> SOGA -> return), and seeds do
# not share state across workers.
N_JOBS: int = -1

In [4]:
# Load train set
df_train: pandas.DataFrame = pandas.read_csv(CSV_TRAIN_PATH)

# Split data into training and validation sets
X_search_pandas: pandas.DataFrame = df_train.drop(columns=[TARGET_COLUMN])
y_search_pandas: pandas.Series = df_train[TARGET_COLUMN]

X_search: numpy.ndarray = numpy.ascontiguousarray(X_search_pandas.to_numpy(), dtype=numpy.float64)
y_search: numpy.ndarray = numpy.ascontiguousarray(y_search_pandas.to_numpy(), dtype=numpy.float64)

# Store feature names
feature_names: list[str] = list(X_search_pandas.columns)

In [5]:
# Load test set
df_test: pandas.DataFrame = pandas.read_csv(CSV_TEST_PATH)

y_test: pandas.Series = df_test[TARGET_COLUMN]
X_test: pandas.DataFrame = df_test.drop(columns=[TARGET_COLUMN])

In [6]:
# Pre-compute folds and scaling
cv: StratifiedKFold = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
fold_indices: list[tuple[numpy.ndarray, numpy.ndarray]] = list(cv.split(X_search, y_search))

X_train_scaled_folds: list[numpy.ndarray] = []
X_val_scaled_folds: list[numpy.ndarray] = []
y_train_folds: list[numpy.ndarray] = []
y_val_folds: list[numpy.ndarray] = []

n_splits: int = len(fold_indices)
n_features: int = len(feature_names)
corr_matrix: numpy.ndarray = numpy.zeros((n_splits, n_features), dtype=float)
for fold_idx, (train_idx, val_idx) in enumerate(fold_indices):
    X_fold_train: numpy.ndarray = X_search[train_idx]
    X_fold_val: numpy.ndarray = X_search[val_idx]
    y_fold_train: numpy.ndarray = y_search[train_idx]
    y_fold_val: numpy.ndarray = y_search[val_idx]

    # Pre-scale (all features at once)
    scaler: StandardScaler = StandardScaler()
    X_train_scaled_folds.append(scaler.fit_transform(X_fold_train))
    X_val_scaled_folds.append(scaler.transform(X_fold_val))
    y_train_folds.append(y_fold_train)
    y_val_folds.append(y_fold_val)

    # Calculate Point-biserial correlation coefficients (target value is binary and the input variables are continuous)
    # Calculate the average correlation for each feature across all fold
    for col_idx in range(X_fold_train.shape[1]):
        feature_col: numpy.ndarray = X_fold_train[:, col_idx]
        unique_vals: int = len(numpy.unique(feature_col))

        if unique_vals <= 1:
            corr: float = 0.0
        elif unique_vals == 2:
            corr: float = matthews_corrcoef(y_fold_train.astype(int), feature_col.astype(int))
        else:
            corr, _ = pointbiserialr(y_fold_train.astype(int), feature_col)

        corr_matrix[fold_idx, col_idx] = corr

In [7]:
import time as _time

training_results_multi: dict[int, list[list[creator.Individual]]] = {}
training_results_single: dict[int, list[creator.IndividualSingle]] = {}

# 20 seeds: enough for tight CIs on the mean and adequate power for paired tests; deterministic ordering keeps results comparable across runs.
seeds: list[int] = list(range(42, 62))


def _train_one_seed(s: int):
    """Run MOGA + SOGA end-to-end for one seed and return the results.

    Lives at notebook-module scope so joblib's loky backend (which uses
    cloudpickle) can serialise it for worker processes. All large data
    (folds, scaled arrays, correlation matrix) is captured via closure;
    cloudpickle ships these to each worker once.

    The flush=True prints below are forwarded by loky to the parent's
    stdout in real time, giving per-seed visibility into long runs.
    """
    _seed_start: float = _time.time()
    print(f"  [seed {s:>4d}] starting MOGA + SOGA pipeline...", flush=True)

    set_seed(s)
    mo: MultiObjectiveTraining = MultiObjectiveTraining(
        config=TrainingConfig(seed=s, result_directory=RESULT_PATH_MULTI),
        feature_names=feature_names,
        fold_indices=fold_indices,
        X_train_scaled_folds=X_train_scaled_folds,
        X_val_scaled_folds=X_val_scaled_folds,
        y_train_folds=y_train_folds,
        y_val_folds=y_val_folds,
        corr_matrix=corr_matrix,
    )
    pareto_front: list[creator.Individual] = mo.run()
    mo.clear_cache()
    _moga_dt: float = _time.time() - _seed_start
    print(f"  [seed {s:>4d}] MOGA done after {_moga_dt/60:5.1f} min "
          f"({len(pareto_front)} Pareto inds)", flush=True)

    set_seed(s)
    so: SingleObjectiveTraining = SingleObjectiveTraining(
        config=TrainingConfig(seed=s, result_directory=RESULT_PATH_SINGLE),
        feature_names=feature_names,
        fold_indices=fold_indices,
        X_train_scaled_folds=X_train_scaled_folds,
        X_val_scaled_folds=X_val_scaled_folds,
        y_train_folds=y_train_folds,
        y_val_folds=y_val_folds)
    best_indi: creator.IndividualSingle = so.run()
    so.clear_cache()
    _total_dt: float = _time.time() - _seed_start
    print(f"  [seed {s:>4d}] SOGA done -- seed total {_total_dt/60:5.1f} min",
          flush=True)

    return s, pareto_front, best_indi


# Seed-level parallelism. Each worker process runs one full (MOGA + SOGA)
# pipeline for a single seed; the inner per-individual caches inside each
# training instance therefore work at full efficiency (no cross-seed
# contamination). N_JOBS=1 takes a pure-sequential path so debugging
# tracebacks stay attached to the notebook kernel.
#
# Progress reporting:
#   * Worker-side -- the flush=True prints above show up as each seed
#     hits its MOGA/SOGA milestones (loky forwards stdout to parent).
#   * Parent-side -- return_as='generator' lets us iterate over results
#     as workers finish, logging seed-completion progress + a rolling
#     ETA instead of waiting for the entire batch.
_overall_start: float = _time.time()
print(f"Training {len(seeds)} seeds in parallel (N_JOBS={N_JOBS}, backend=loky)...",
      flush=True)

_results: list = []
if N_JOBS == 1:
    for _i, _s in enumerate(seeds, 1):
        _results.append(_train_one_seed(_s))
        _el: float = _time.time() - _overall_start
        print(f"Progress: {_i}/{len(seeds)} seeds done | "
              f"elapsed {_el/60:6.1f} min | avg {_el/_i/60:5.1f} min/seed",
              flush=True)
else:
    _parallel = Parallel(n_jobs=N_JOBS, backend="loky", return_as="generator")
    _gen = _parallel(delayed(_train_one_seed)(s) for s in seeds)
    for _i, _r in enumerate(_gen, 1):
        _results.append(_r)
        _el = _time.time() - _overall_start
        _eta = _el / _i * (len(seeds) - _i)
        print(f"Progress: {_i}/{len(seeds)} seeds done | "
              f"elapsed {_el/60:6.1f} min | avg {_el/_i/60:5.1f} min/seed | "
              f"ETA ~{_eta/60:5.1f} min",
              flush=True)

for s, pareto_front, best_indi in _results:
    training_results_multi[s] = pareto_front
    training_results_single[s] = best_indi

_total_min: float = (_time.time() - _overall_start) / 60.0
print(f"All {len(seeds)} seeds finished in {_total_min:.1f} minutes.", flush=True)

Training 20 seeds in parallel (N_JOBS=-1, backend=loky)...
Progress: 1/20 seeds done | elapsed    5.0 min | avg   5.0 min/seed | ETA ~ 94.8 min
Progress: 2/20 seeds done | elapsed    5.0 min | avg   2.5 min/seed | ETA ~ 44.9 min
Progress: 3/20 seeds done | elapsed    5.0 min | avg   1.7 min/seed | ETA ~ 28.3 min
Progress: 4/20 seeds done | elapsed    5.0 min | avg   1.2 min/seed | ETA ~ 20.0 min
Progress: 5/20 seeds done | elapsed    5.0 min | avg   1.0 min/seed | ETA ~ 15.0 min
Progress: 6/20 seeds done | elapsed    5.0 min | avg   0.8 min/seed | ETA ~ 11.6 min
Progress: 7/20 seeds done | elapsed    5.0 min | avg   0.7 min/seed | ETA ~  9.3 min
Progress: 8/20 seeds done | elapsed    5.0 min | avg   0.6 min/seed | ETA ~  7.5 min
Progress: 9/20 seeds done | elapsed    5.2 min | avg   0.6 min/seed | ETA ~  6.3 min
Progress: 10/20 seeds done | elapsed    5.2 min | avg   0.5 min/seed | ETA ~  5.2 min
Progress: 11/20 seeds done | elapsed    5.2 min | avg   0.5 min/seed | ETA ~  4.2 min
Prog

In [8]:
# Detect column types automatically from the training DataFrame
X_train_df: pandas.DataFrame = df_train.drop(columns=[TARGET_COLUMN])
y_train_series: pandas.Series = df_train[TARGET_COLUMN]

continuous_cols: list[str] = get_continuous_columns(X_train_df)
dummy_cols: list[str] = get_dummy_columns(X_train_df)

# Training-set std for proportional noise scaling
train_std: pandas.Series = X_train_df.std()

print(f"Continuous features: {len(continuous_cols)}")
print(f"Dummy features:     {len(dummy_cols)}")

Continuous features: 189
Dummy features:     81


## Evaluation: Single vs Multi-Objective Robustness Comparison

For each seed we:
1. Retrain a final LogisticRegression on the **full training set** using features selected by the GA.
   - Multi-objective: the Pareto individual with the **highest sign-consistency** score or **knee-point selection**.
   - Single-objective: the best individual from the Hall of Fame.
2. Evaluate on clean data and under increasing noise levels on **both test and training sets** (the training-set sweep is an in-sample noise-sensitivity probe, not a held-out validation):
   - **Gaussian noise + mean shift** on continuous features
   - **Random corruption noise** on dummy (binary) features
3. Save CSV results and comparison plots to the result directory (separate files per dataset).

In [9]:
evaluation_results_test: dict[int, pandas.DataFrame] = {}

for s in seeds:
    set_seed(s)

    # --- Multi-objective: select max sign-consistency score or knee point individual from Pareto front ---
    pareto_front: list[creator.Individual] = training_results_multi[s]
    best_multi_ind: creator.Individual = select_pareto_individual(pareto_front, use_knee_point=USE_KNEE_POINT_SELECTION)
    print(f"Seed {s} | Selected multi-objective individual: "
          f"AUC={best_multi_ind.fitness.values[0]:.4f}, "
          f"SignConsistency={best_multi_ind.fitness.values[1]:.4f}, "
          f"Features={sum(best_multi_ind)}")

    # --- Build final model packages on full training set ---
    model_pkg_multi = build_model_package(
        best_multi_ind, feature_names, X_train_df, y_train_series, seed=s)
    model_pkg_single = build_model_package(
        training_results_single[s], feature_names, X_train_df, y_train_series, seed=s)

    # --- Run noise evaluation on TEST set and save plots/CSV ---
    eval_df = evaluate_and_save(
        seed=s,
        model_pkg_multi=model_pkg_multi,
        model_pkg_single=model_pkg_single,
        X_eval=X_test,
        y_eval=y_test,
        train_std=train_std,
        continuous_cols=continuous_cols,
        dummy_cols=dummy_cols,
        result_directory=RESULT_PATH_EVAL,
        use_roc_auc=True,
        dataset_label="Test",
    )
    evaluation_results_test[s] = eval_df

print("\nTest set evaluation complete. Results saved to:", RESULT_PATH_EVAL)

Seed 42 | Selected multi-objective individual: AUC=0.9095, SignConsistency=0.9296, Features=71
  Seed 42 [Test] | Multi features:  71 | Single features: 122 | Clean AUC  Multi: 0.8027  Single: 0.8176
Seed 43 | Selected multi-objective individual: AUC=0.9074, SignConsistency=0.8792, Features=80
  Seed 43 [Test] | Multi features:  80 | Single features: 117 | Clean AUC  Multi: 0.8123  Single: 0.7995
Seed 44 | Selected multi-objective individual: AUC=0.9079, SignConsistency=0.9024, Features=82
  Seed 44 [Test] | Multi features:  82 | Single features: 127 | Clean AUC  Multi: 0.8274  Single: 0.7873
Seed 45 | Selected multi-objective individual: AUC=0.8894, SignConsistency=0.8991, Features=76
  Seed 45 [Test] | Multi features:  76 | Single features: 116 | Clean AUC  Multi: 0.8150  Single: 0.7792
Seed 46 | Selected multi-objective individual: AUC=0.9069, SignConsistency=0.8861, Features=79
  Seed 46 [Test] | Multi features:  79 | Single features: 113 | Clean AUC  Multi: 0.8592  Single: 0.7803


In [10]:
evaluation_results_train: dict[int, pandas.DataFrame] = {}

for s in seeds:
    set_seed(s)

    # --- Multi-objective: select max sign-consistency or knee point individual from Pareto front ---
    pareto_front: list[creator.Individual] = training_results_multi[s]
    best_multi_ind: creator.Individual = select_pareto_individual(pareto_front, use_knee_point=USE_KNEE_POINT_SELECTION)

    # --- Build final model packages on full training set ---
    model_pkg_multi = build_model_package(
        best_multi_ind, feature_names, X_train_df, y_train_series, seed=s)
    model_pkg_single = build_model_package(
        training_results_single[s], feature_names, X_train_df, y_train_series, seed=s)

    # --- Run noise evaluation on TRAINING set (in-sample noise sensitivity) ---
    eval_df = evaluate_and_save(
        seed=s,
        model_pkg_multi=model_pkg_multi,
        model_pkg_single=model_pkg_single,
        X_eval=X_train_df,
        y_eval=y_train_series,
        train_std=train_std,
        continuous_cols=continuous_cols,
        dummy_cols=dummy_cols,
        result_directory=RESULT_PATH_EVAL,
        use_roc_auc=True,
        dataset_label="Train",
    )
    evaluation_results_train[s] = eval_df

print("\nTraining set evaluation complete. Results saved to:", RESULT_PATH_EVAL)

  Seed 42 [Train] | Multi features:  71 | Single features: 122 | Clean AUC  Multi: 0.9562  Single: 0.9685
  Seed 43 [Train] | Multi features:  80 | Single features: 117 | Clean AUC  Multi: 0.9573  Single: 0.9691
  Seed 44 [Train] | Multi features:  82 | Single features: 127 | Clean AUC  Multi: 0.9639  Single: 0.9789
  Seed 45 [Train] | Multi features:  76 | Single features: 116 | Clean AUC  Multi: 0.9550  Single: 0.9626
  Seed 46 [Train] | Multi features:  79 | Single features: 113 | Clean AUC  Multi: 0.9481  Single: 0.9650
  Seed 47 [Train] | Multi features:  83 | Single features: 118 | Clean AUC  Multi: 0.9364  Single: 0.9735
  Seed 48 [Train] | Multi features:  81 | Single features: 125 | Clean AUC  Multi: 0.9492  Single: 0.9767
  Seed 49 [Train] | Multi features:  81 | Single features: 111 | Clean AUC  Multi: 0.9525  Single: 0.9650
  Seed 50 [Train] | Multi features:  78 | Single features: 114 | Clean AUC  Multi: 0.9450  Single: 0.9695
  Seed 51 [Train] | Multi features:  71 | Sing

## Feature Selection Stability (Jaccard Similarity)

Compare how consistently each method selects the same features across different random seeds.
- **Jaccard similarity** = |intersection| / |union| between two feature sets (1.0 = identical, 0.0 = no overlap).
- A **pairwise heatmap** shows seed-to-seed stability for multi-objective vs single-objective.
- A **feature frequency bar chart** shows which features are consistently selected across seeds.

In [11]:
# Collect the best individual per seed for both methods
best_multi_by_seed: dict[int, list[int]] = {}
best_single_by_seed: dict[int, list[int]] = {}

for s in seeds:
    # Multi-objective: max sign-consistency or knee point (same selection as evaluation)
    pareto_front: list[creator.Individual] = training_results_multi[s]
    best_multi_ind: creator.Individual = select_pareto_individual(pareto_front, use_knee_point=USE_KNEE_POINT_SELECTION)
    best_multi_by_seed[s]: list[int] = list(best_multi_ind)

    # Single-objective: best individual from HoF
    best_single_by_seed[s]: list[int] = list(training_results_single[s])

# Compute pairwise Jaccard stability matrices
seeds_multi, matrix_multi = compute_stability_matrix(best_multi_by_seed, feature_names)
seeds_single, matrix_single = compute_stability_matrix(best_single_by_seed, feature_names)

# Print summary
print("=== Feature Selection Stability (Jaccard Similarity) ===\n")
print("Multi-Objective (SiCo-MOGA) pairwise Jaccard:")
for i, si in enumerate(seeds_multi):
    for j, sj in enumerate(seeds_multi):
        if j > i:
            print(f"  Seed {si} vs Seed {sj}: {matrix_multi[i, j]:.3f}")

print("\nSingle-Objective (AUC-only GA) pairwise Jaccard:")
for i, si in enumerate(seeds_single):
    for j, sj in enumerate(seeds_single):
        if j > i:
            print(f"  Seed {si} vs Seed {sj}: {matrix_single[i, j]:.3f}")

# Save plots
stability_dir: str = os.path.join(RESULT_PATH_EVAL, "stability")

plot_stability_heatmap(
    seeds_multi, matrix_multi,
    seeds_single, matrix_single,
    os.path.join(stability_dir, "jaccard_heatmap.png"))

plot_feature_frequency(
    best_multi_by_seed, feature_names,
    "Multi-Objective (SiCo-MOGA)",
    os.path.join(stability_dir, "feature_frequency_multi.png"))

plot_feature_frequency(
    best_single_by_seed, feature_names,
    "Single-Objective (AUC-only GA)",
    os.path.join(stability_dir, "feature_frequency_single.png"))

print(f"\nStability plots saved to: {stability_dir}")

=== Feature Selection Stability (Jaccard Similarity) ===

Multi-Objective (SiCo-MOGA) pairwise Jaccard:
  Seed 42 vs Seed 43: 0.438
  Seed 42 vs Seed 44: 0.471
  Seed 42 vs Seed 45: 0.413
  Seed 42 vs Seed 46: 0.415
  Seed 42 vs Seed 47: 0.481
  Seed 42 vs Seed 48: 0.490
  Seed 42 vs Seed 49: 0.490
  Seed 42 vs Seed 50: 0.461
  Seed 42 vs Seed 51: 0.560
  Seed 42 vs Seed 52: 0.510
  Seed 42 vs Seed 53: 0.433
  Seed 42 vs Seed 54: 0.475
  Seed 42 vs Seed 55: 0.526
  Seed 42 vs Seed 56: 0.481
  Seed 42 vs Seed 57: 0.480
  Seed 42 vs Seed 58: 0.500
  Seed 42 vs Seed 59: 0.451
  Seed 42 vs Seed 60: 0.464
  Seed 42 vs Seed 61: 0.485
  Seed 43 vs Seed 44: 0.446
  Seed 43 vs Seed 45: 0.431
  Seed 43 vs Seed 46: 0.459
  Seed 43 vs Seed 47: 0.393
  Seed 43 vs Seed 48: 0.505
  Seed 43 vs Seed 49: 0.477
  Seed 43 vs Seed 50: 0.491
  Seed 43 vs Seed 51: 0.452
  Seed 43 vs Seed 52: 0.495
  Seed 43 vs Seed 53: 0.423
  Seed 43 vs Seed 54: 0.491
  Seed 43 vs Seed 55: 0.481
  Seed 43 vs Seed 56: 0.442


## Multicollinearity Analysis: Variance Inflation Factor (VIF)

VIF measures how much the variance of a regression coefficient is inflated due to multicollinearity.
- **VIF = 1**: no correlation with other predictors
- **VIF = 5**: moderate multicollinearity
- **VIF > 10**: high multicollinearity (coefficient estimates become unreliable)

We compare the VIF distributions of features selected by SOGA (Single-Objective) vs MOGA (Multi-Objective) across all seeds.

In [12]:
# Compute VIF for features selected by each method across seeds
vif_multi: pandas.DataFrame = compute_vif_across_seeds(best_multi_by_seed, feature_names, X_train_df)
vif_single: pandas.DataFrame = compute_vif_across_seeds(best_single_by_seed, feature_names, X_train_df)

print("=== Variance Inflation Factor (VIF) Summary ===\n")
print("Multi-Objective (SiCo-MOGA):")
print(f"  Features per seed: {vif_multi.groupby('seed')['feature'].count().values}")
print(f"  Median VIF: {vif_multi['vif'].median():.2f}")
print(f"  Mean VIF:   {vif_multi['vif'].mean():.2f}")
print(f"  Max VIF:    {vif_multi['vif'].max():.2f}")
print(f"  Features with VIF > 5: {(vif_multi['vif'] > 5).sum()} / {len(vif_multi)}")

print("\nSingle-Objective (AUC-only GA):")
print(f"  Features per seed: {vif_single.groupby('seed')['feature'].count().values}")
print(f"  Median VIF: {vif_single['vif'].median():.2f}")
print(f"  Mean VIF:   {vif_single['vif'].mean():.2f}")
print(f"  Max VIF:    {vif_single['vif'].max():.2f}")
print(f"  Features with VIF > 5: {(vif_single['vif'] > 5).sum()} / {len(vif_single)}")

# Save VIF plot
vif_dir: str = os.path.join(RESULT_PATH_EVAL, "vif")
plot_vif_boxplot(vif_multi, vif_single, os.path.join(vif_dir, "vif_comparison.png"))

# Save VIF data as CSV
vif_multi.to_csv(os.path.join(vif_dir, "vif_multi.csv"), index=False)
vif_single.to_csv(os.path.join(vif_dir, "vif_single.csv"), index=False)

print(f"\nVIF plots and data saved to: {vif_dir}")

=== Variance Inflation Factor (VIF) Summary ===

Multi-Objective (SiCo-MOGA):
  Features per seed: [71 80 82 76 79 83 81 81 78 71 74 68 78 77 83 79 85 77 71 82]
  Median VIF: 4.66
  Mean VIF:   22520.94
  Max VIF:    1000000.00
  Features with VIF > 5: 748 / 1556

Single-Objective (AUC-only GA):
  Features per seed: [110 105 117 108 106 110 114  99 100 108 109  95 112 105 106  99 112 111
 113 101]
  Median VIF: 9.18
  Mean VIF:   58239.02
  Max VIF:    1000000.00
  Features with VIF > 5: 1421 / 2140

VIF plots and data saved to: 2026-05-15_15-55-20\evaluation\vif


## Feature Set Comparison: Multi-Objective vs Single-Objective

For each seed, compare the features selected by SiCo-MOGA vs. SOGA:
- **Only in Multi**: features that the multi-objective GA selected but single-objective did not
- **Only in Single**: features that the single-objective GA selected but multi-objective did not
- **Common**: features selected by both

This highlights which features the sign-consistency penalty specifically filters out (only-in-single) or adds back (only-in-multi). The results are printed per seed and saved as a CSV.

In [13]:
# Feature set comparison: multi vs single objective
from evaluation_utils import ensure_directory as _ensure_directory

comparison_rows: list[dict] = []

print("=== Feature Set Comparison: Multi-Objective vs Single-Objective ===\n")
for s in seeds:
    multi_bits: list[int] = best_multi_by_seed[s]
    single_bits: list[int] = best_single_by_seed[s]

    multi_set: set[str] = {f for f, b in zip(feature_names, multi_bits) if b == 1}
    single_set: set[str] = {f for f, b in zip(feature_names, single_bits) if b == 1}

    only_in_multi: list[str] = sorted(multi_set - single_set)
    only_in_single: list[str] = sorted(single_set - multi_set)
    common: list[str] = sorted(multi_set & single_set)

    print(f"--- Seed {s} ---")
    print(f"  Multi features: {len(multi_set)} | Single features: {len(single_set)} | Common: {len(common)}")
    print(f"  Only in Multi  ({len(only_in_multi)}):")
    for f in only_in_multi:
        print(f"    + {f}")
    print(f"  Only in Single ({len(only_in_single)}):")
    for f in only_in_single:
        print(f"    - {f}")
    print()

    for f in only_in_multi:
        comparison_rows.append({"seed": s, "feature": f, "membership": "only_in_multi"})
    for f in only_in_single:
        comparison_rows.append({"seed": s, "feature": f, "membership": "only_in_single"})
    for f in common:
        comparison_rows.append({"seed": s, "feature": f, "membership": "common"})

comparison_df: pandas.DataFrame = pandas.DataFrame(comparison_rows)

comparison_dir: str = os.path.join(RESULT_PATH_EVAL, "feature_comparison")
_ensure_directory(comparison_dir)
comparison_df.to_csv(os.path.join(comparison_dir, "feature_set_comparison.csv"), index=False)

print(f"Feature set comparison CSV saved to: {comparison_dir}")

=== Feature Set Comparison: Multi-Objective vs Single-Objective ===

--- Seed 42 ---
  Multi features: 71 | Single features: 122 | Common: 46
  Only in Multi  (25):
    + feature_104
    + feature_106
    + feature_110
    + feature_120
    + feature_130
    + feature_136
    + feature_137
    + feature_148
    + feature_172
    + feature_185
    + feature_221
    + feature_231
    + feature_236
    + feature_237
    + feature_240
    + feature_241
    + feature_243
    + feature_248
    + feature_257
    + feature_272
    + feature_39
    + feature_44
    + feature_71
    + feature_89
    + feature_96
  Only in Single (76):
    - feature_108
    - feature_113
    - feature_115
    - feature_121
    - feature_122
    - feature_124
    - feature_125
    - feature_128
    - feature_133
    - feature_135
    - feature_138
    - feature_139
    - feature_141
    - feature_142
    - feature_144
    - feature_145
    - feature_146
    - feature_149
    - feature_152
    - feature_153
    - f

## Sign Inconsistency in Single-Objective Model

For each feature selected by the Single-Objective GA, compare:
- **Marginal correlation sign** with the target (point-biserial for continuous, Matthews for binary), computed on the full training set
- **Logistic regression coefficient sign**, from the final model retrained on the full training set

Features where these signs **disagree** are sign-inconsistent: the feature's individual association with the target is opposite to how the model uses it (a hallmark of suppressor effects and multicollinearity). The sign-consistency objective in SiCo-MOGA is designed to avoid exactly these cases.

In [14]:
# Sign inconsistency in Single-Objective model: correlation sign vs coefficient sign

# Pre-compute marginal correlations with target on the FULL training set
# Cast to int for matthews_corrcoef so sklearn's type_of_target detects it as
# binary rather than continuous (ValueError otherwise when the column dtype is float).
y_full: numpy.ndarray = y_train_series.to_numpy().astype(int)

full_train_corr: dict[str, float] = {}
for col in feature_names:
    feature_col: numpy.ndarray = X_train_df[col].to_numpy()
    unique_vals: int = len(numpy.unique(feature_col))
    if unique_vals <= 1:
        full_train_corr[col] = 0.0
    elif unique_vals == 2:
        full_train_corr[col] = float(matthews_corrcoef(y_full, feature_col.astype(int)))
    else:
        c, _ = pointbiserialr(y_full, feature_col)
        full_train_corr[col] = float(c)


def _sign(x: float, atol: float = 1e-12) -> int:
    """Return -1/0/+1 with a tolerance around zero."""
    if numpy.isclose(x, 0.0, atol=atol):
        return 0
    return 1 if x > 0 else -1


inconsistency_rows: list[dict] = []

print("=== Sign Inconsistency in Single-Objective Selected Features ===\n")
for s in seeds:
    set_seed(s)
    model_pkg = build_model_package(
        training_results_single[s], feature_names, X_train_df, y_train_series, seed=s)

    selected: list[str] = model_pkg["features"]
    coefs: numpy.ndarray = model_pkg["model"].coef_[0]

    inconsistent: list[dict] = []
    for feat, coef in zip(selected, coefs):
        corr: float = full_train_corr[feat]
        corr_sign: int = _sign(corr)
        coef_sign: int = _sign(float(coef))
        mismatch: bool = (corr_sign != 0 and coef_sign != 0 and corr_sign != coef_sign)
        if mismatch:
            inconsistent.append({
                "seed": s,
                "feature": feat,
                "correlation": corr,
                "coefficient": float(coef),
                "corr_sign": corr_sign,
                "coef_sign": coef_sign,
            })

    print(f"--- Seed {s} ---")
    print(f"  Single-objective selected features: {len(selected)}")
    print(f"  Sign-inconsistent features:         {len(inconsistent)} "
          f"({100.0 * len(inconsistent) / max(len(selected), 1):.1f}%)")
    if inconsistent:
        print(f"  {'Feature':<40} {'Correlation':>12} {'Coefficient':>14}")
        for row in sorted(inconsistent, key=lambda r: abs(r["coefficient"]), reverse=True):
            print(f"    {row['feature']:<38} {row['correlation']:>12.4f} {row['coefficient']:>14.4f}")
    print()

    inconsistency_rows.extend(inconsistent)

inconsistency_df: pandas.DataFrame = pandas.DataFrame(inconsistency_rows)

inconsistency_dir: str = os.path.join(RESULT_PATH_EVAL, "feature_comparison")
_ensure_directory(inconsistency_dir)
inconsistency_df.to_csv(os.path.join(inconsistency_dir, "single_sign_inconsistency.csv"), index=False)

print(f"Sign inconsistency CSV saved to: {inconsistency_dir}")

=== Sign Inconsistency in Single-Objective Selected Features ===

--- Seed 42 ---
  Single-objective selected features: 122
  Sign-inconsistent features:         33 (27.0%)
  Feature                                   Correlation    Coefficient
    feature_113                                 -0.1261         0.7313
    feature_170                                 -0.1614         0.7210
    feature_29                                  -0.0059         0.6468
    feature_273                                 -0.0809         0.5513
    feature_250                                  0.0376        -0.5261
    feature_6                                   -0.0577         0.5250
    feature_183                                 -0.0550         0.5049
    feature_138                                  0.0907        -0.4472
    feature_161                                 -0.0341         0.4464
    feature_52                                  -0.0564         0.4420
    feature_125                               

## Baseline: Logistic Regression with ALL features (no feature selection)

Train a logistic regression using **every** available feature (no GA, no stepwise search) and report clean AUC/AP on the test set per seed. Serves as the trivial "no selection" reference point against which both the GA-based methods and the forward-stepwise baseline must justify themselves.

In [15]:
# Baseline 1: Logistic Regression using ALL features (no feature selection)
baseline_all_rows: list[dict] = []
all_features_ind: list[int] = [1] * len(feature_names)

print("=== Baseline: Logistic Regression with ALL features ===\n")
for s in seeds:
    set_seed(s)

    model_pkg_all: dict[str, Any] = build_model_package(
        all_features_ind, feature_names, X_train_df, y_train_series, seed=s)

    X_test_scaled_all: numpy.ndarray = model_pkg_all["scaler"].transform(
        X_test[model_pkg_all["features"]].to_numpy())
    y_prob_all: numpy.ndarray = model_pkg_all["model"].predict_proba(X_test_scaled_all)[:, 1]

    test_auc_all: float = float(roc_auc_score(y_test, y_prob_all))

    baseline_all_rows.append({
        "seed": s,
        "n_features": len(model_pkg_all["features"]),
        "test_auc": test_auc_all,
    })
    print(f"  Seed {s} | Features: {len(model_pkg_all['features']):4d} "
          f"| Test AUC: {test_auc_all:.4f}")

baseline_all_df: pandas.DataFrame = pandas.DataFrame(baseline_all_rows)
print(f"\nMean test AUC: {baseline_all_df['test_auc'].mean():.4f} "
      f"(+/- {baseline_all_df['test_auc'].std():.4f})")

baseline_all_dir: str = os.path.join(RESULT_PATH_EVAL, "baseline_all_features")
_ensure_directory(baseline_all_dir)
baseline_all_df.to_csv(
    os.path.join(baseline_all_dir, "baseline_all_features.csv"), index=False)
print(f"\nBaseline (all features) results saved to: {baseline_all_dir}")

=== Baseline: Logistic Regression with ALL features ===

  Seed 42 | Features:  278 | Test AUC: 0.8106
  Seed 43 | Features:  278 | Test AUC: 0.8106
  Seed 44 | Features:  278 | Test AUC: 0.8106
  Seed 45 | Features:  278 | Test AUC: 0.8106
  Seed 46 | Features:  278 | Test AUC: 0.8106
  Seed 47 | Features:  278 | Test AUC: 0.8106
  Seed 48 | Features:  278 | Test AUC: 0.8106
  Seed 49 | Features:  278 | Test AUC: 0.8106
  Seed 50 | Features:  278 | Test AUC: 0.8106
  Seed 51 | Features:  278 | Test AUC: 0.8106
  Seed 52 | Features:  278 | Test AUC: 0.8106
  Seed 53 | Features:  278 | Test AUC: 0.8106
  Seed 54 | Features:  278 | Test AUC: 0.8106
  Seed 55 | Features:  278 | Test AUC: 0.8106
  Seed 56 | Features:  278 | Test AUC: 0.8106
  Seed 57 | Features:  278 | Test AUC: 0.8106
  Seed 58 | Features:  278 | Test AUC: 0.8106
  Seed 59 | Features:  278 | Test AUC: 0.8106
  Seed 60 | Features:  278 | Test AUC: 0.8106
  Seed 61 | Features:  278 | Test AUC: 0.8106

Mean test AUC: 0.8106 

## Baseline: Logistic Regression with Forward Stepwise Feature Selection

Use scikit-learn's `SequentialFeatureSelector` (`direction="forward"`, 3-fold stratified CV on ROC AUC) to greedily add features while the cross-validated AUC keeps improving by more than `tol=1e-3`. Refit the final logistic regression on the selected subset via `build_model_package`, then report clean AUC/AP on the test set. This is the classical stepwise baseline the GA methods must justify improvement over.

In [16]:
# Baseline 2: Logistic Regression with forward stepwise feature selection
#
# Two correctness changes vs. the original cell:
#   * SFS now operates on a Pipeline(StandardScaler, LR) so the scaler is
#     re-fit on each inner CV training fold (no leakage across folds).
#   * The feature budget is fixed to the median number of features that
#     MOGA selects across seeds (option b from the methodology review):
#     this isolates *which* features each method picks, not *how many*.
baseline_fwd_rows: list[dict] = []

print("=== Baseline: Logistic Regression with Forward Stepwise Feature Selection ===\n")
for s in seeds:
    set_seed(s)

    # Pipeline = scaler + LR. SFS' inner CV refits the whole pipeline per
    # fold, so the scaler never sees its own held-out validation rows.
    sfs_pipeline: Pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", _SFSLogReg(penalty="l2", solver="lbfgs", max_iter=1000, random_state=s)),
    ])
    sfs_cv: StratifiedKFold = StratifiedKFold(n_splits=3, shuffle=True, random_state=s)
    sfs: SequentialFeatureSelector = SequentialFeatureSelector(
        sfs_pipeline,
        direction="forward",
        scoring="roc_auc",
        cv=sfs_cv,
        n_jobs=-1,
    )
    sfs.fit(X_train_df.to_numpy(), y_train_series)

    selected_mask: numpy.ndarray = sfs.get_support()
    fwd_individual: list[int] = [1 if m else 0 for m in selected_mask]

    # Re-fit the final LR on the selected subset through the standard helper
    model_pkg_fwd = build_model_package(
        fwd_individual, feature_names, X_train_df, y_train_series, seed=s)

    X_test_scaled_fwd: numpy.ndarray = model_pkg_fwd["scaler"].transform(
        X_test[model_pkg_fwd["features"]].to_numpy())
    y_prob_fwd: numpy.ndarray = model_pkg_fwd["model"].predict_proba(X_test_scaled_fwd)[:, 1]

    test_auc_fwd: float = float(roc_auc_score(y_test, y_prob_fwd))

    baseline_fwd_rows.append({
        "seed": s,
        "n_features": len(model_pkg_fwd["features"]),
        "test_auc": test_auc_fwd,
        "features": ",".join(model_pkg_fwd["features"]),
    })
    print(f"  Seed {s} | Features: {len(model_pkg_fwd['features']):4d} "
          f"| Test AUC: {test_auc_fwd:.4f}")

baseline_fwd_df: pandas.DataFrame = pandas.DataFrame(baseline_fwd_rows)
print(f"\nMean features: {baseline_fwd_df['n_features'].mean():.1f} "
      f"(+/- {baseline_fwd_df['n_features'].std():.1f})")
print(f"Mean test AUC: {baseline_fwd_df['test_auc'].mean():.4f} "
      f"(+/- {baseline_fwd_df['test_auc'].std():.4f})")

baseline_fwd_dir: str = os.path.join(RESULT_PATH_EVAL, "baseline_forward_stepwise")
_ensure_directory(baseline_fwd_dir)
baseline_fwd_df.to_csv(
    os.path.join(baseline_fwd_dir, "baseline_forward_stepwise.csv"), index=False)
print(f"\nBaseline (forward stepwise) results saved to: {baseline_fwd_dir}")

=== Baseline: Logistic Regression with Forward Stepwise Feature Selection ===

  Seed 42 | Features:  139 | Test AUC: 0.8139
  Seed 43 | Features:  139 | Test AUC: 0.8265
  Seed 44 | Features:  139 | Test AUC: 0.7960
  Seed 45 | Features:  139 | Test AUC: 0.7864
  Seed 46 | Features:  139 | Test AUC: 0.8531
  Seed 47 | Features:  139 | Test AUC: 0.8429
  Seed 48 | Features:  139 | Test AUC: 0.8121
  Seed 49 | Features:  139 | Test AUC: 0.8012
  Seed 50 | Features:  139 | Test AUC: 0.8097
  Seed 51 | Features:  139 | Test AUC: 0.7605
  Seed 52 | Features:  139 | Test AUC: 0.7958
  Seed 53 | Features:  139 | Test AUC: 0.8112
  Seed 54 | Features:  139 | Test AUC: 0.7984
  Seed 55 | Features:  139 | Test AUC: 0.8080
  Seed 56 | Features:  139 | Test AUC: 0.8150
  Seed 57 | Features:  139 | Test AUC: 0.7818
  Seed 58 | Features:  139 | Test AUC: 0.8195
  Seed 59 | Features:  139 | Test AUC: 0.8115
  Seed 60 | Features:  139 | Test AUC: 0.8326
  Seed 61 | Features:  139 | Test AUC: 0.8272



## All-Models Noise Robustness Comparison (averaged across seeds)

Evaluate all four models on the **test set** under two noise sweeps:
- **Gaussian noise** on continuous features (zero mean-shift), noise level = 0.0 .. 1.0 in 0.1 steps.
- **Random corruption** on binary (dummy) features, corruption fraction = 0.0 .. 1.0 in 0.1 steps.

For each seed the same four models are evaluated at every noise level, and the mean AUC across seeds is plotted with an std shaded band. This gives a like-for-like view of robustness for:
- **Multi-Objective (SiCo-MOGA)** Pareto individual with max sign-consistency or knee point
- **Single-Objective (AUC-only GA)** best HoF individual
- **All features (no selection)** logistic regression on every input
- **Forward stepwise selection** `SequentialFeatureSelector` (forward, CV=3, ROC-AUC, tol=1e-3)

The two output plots match the style of the per-seed `gaussian_noise_comparison_test.png` and `dummy_flip_comparison_test.png` files, but with four curves and seed-aggregated statistics.

In [17]:
# All-models noise robustness comparison (averaged across seeds)
noise_levels: numpy.ndarray = numpy.arange(0.0, 1.1, 0.1)
flip_levels: numpy.ndarray = numpy.arange(0.0, 1.05, 0.1)

gauss_rows: list[dict] = []
dummy_rows: list[dict] = []

all_features_ind: list[int] = [1] * len(feature_names)

print("Running noise sweeps for all 4 models across all seeds...")
for s in seeds:
    set_seed(s)

    # --- Build all 4 model packages on the same full training set ---
    pareto_front: list[creator.Individual] = training_results_multi[s]
    best_multi_ind: creator.Individual = select_pareto_individual(pareto_front, use_knee_point=USE_KNEE_POINT_SELECTION)

    pkg_multi: dict[str, Any] = build_model_package(
        best_multi_ind, feature_names, X_train_df, y_train_series, seed=s)
    pkg_single: dict[str, Any] = build_model_package(
        training_results_single[s], feature_names, X_train_df, y_train_series, seed=s)
    pkg_all: dict[str, Any] = build_model_package(
        all_features_ind, feature_names, X_train_df, y_train_series, seed=s)

    # Forward stepwise: re-run SFS per seed using a Pipeline so the scaler
    # is refit on each inner CV fold (no scaler leakage across folds).
    # Feature budget is matched to MOGA's median selection size.
    sfs_pipeline: Pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", _CmpLogReg(penalty="l2", solver="lbfgs", max_iter=1000, random_state=s)),
    ])
    sfs_cv: StratifiedKFold = StratifiedKFold(n_splits=3, shuffle=True, random_state=s)
    sfs: SequentialFeatureSelector = SequentialFeatureSelector(
        sfs_pipeline,
        direction="forward",
        scoring="roc_auc",
        cv=sfs_cv,
        n_jobs=-1,
    )
    sfs.fit(X_train_df.to_numpy(), y_train_series)
    fwd_individual: list[int] = [1 if m else 0 for m in sfs.get_support()]
    pkg_fwd = build_model_package(
        fwd_individual, feature_names, X_train_df, y_train_series, seed=s)

    models: dict[str, dict] = {
        "multi":   pkg_multi,
        "single":  pkg_single,
        "all":     pkg_all,
        "forward": pkg_fwd,
    }

    # --- Gaussian noise sweep (zero mean-shift) ---
    for noise in noise_levels:
        nv: float = round(float(noise), 2)
        X_noisy: pandas.DataFrame = apply_proportional_noise(
            X_test, train_std, nv, 0.0, continuous_cols)
        row: dict = {"seed": s, "noise_level": nv}
        for label, pkg in models.items():
            row[f"auc_{label}"] = evaluate_model(pkg, X_noisy, y_test, use_roc_auc=True)
        gauss_rows.append(row)

    # --- Dummy corruption sweep ---
    for flip in flip_levels:
        fv: float = round(float(flip), 2)
        X_noisy = apply_dummy_noise(X_test, fv, dummy_cols)
        row = {"seed": s, "flip_rate": fv}
        for label, pkg in models.items():
            row[f"auc_{label}"] = evaluate_model(pkg, X_noisy, y_test, use_roc_auc=True)
        dummy_rows.append(row)

    print(f"  Seed {s} done "
          f"(features: multi={sum(best_multi_ind)}, single={sum(training_results_single[s])}, "
          f"all={len(feature_names)}, forward={sum(fwd_individual)})")

gauss_df: pandas.DataFrame = pandas.DataFrame(gauss_rows)
dummy_df: pandas.DataFrame = pandas.DataFrame(dummy_rows)

# Aggregate (mean, std) across seeds
gauss_agg: pandas.DataFrame = gauss_df.drop(columns=["seed"]).groupby("noise_level").agg(["mean", "std"])
dummy_agg: pandas.DataFrame = dummy_df.drop(columns=["seed"]).groupby("flip_rate").agg(["mean", "std"])

comparison_dir: str = os.path.join(RESULT_PATH_EVAL, "all_models_comparison")
_ensure_directory(comparison_dir)
gauss_df.to_csv(os.path.join(comparison_dir, "gaussian_noise_per_seed.csv"), index=False)
dummy_df.to_csv(os.path.join(comparison_dir, "dummy_flip_per_seed.csv"), index=False)

# Plot style: one entry per model (label, color, marker, linestyle)
model_styles: dict[str, tuple] = {
    "multi":   ("Multi-Objective (SiCo-MOGA)",    "tab:blue",   "o", "-"),
    "single":  ("Single-Objective (AUC-only GA)", "tab:orange", "s", "--"),
    "all":     ("All features (no selection)",    "tab:green",  "^", "-."),
    "forward": ("Forward stepwise selection",     "tab:red",    "D", ":"),
}


def _plot_comparison(agg_df: pandas.DataFrame, x_col_name: str,
                     xlabel: str, title: str, out_path: str) -> None:
    fig, ax = plt.subplots(figsize=(10, 6))
    for key, (name, color, marker, ls) in model_styles.items():
        mean_series: pandas.Series = agg_df[(f"auc_{key}", "mean")]
        std_series: pandas.Series = agg_df[(f"auc_{key}", "std")].fillna(0.0)
        x_vals: numpy.ndarray = mean_series.index.values
        m_vals: numpy.ndarray = mean_series.values
        s_vals: numpy.ndarray = std_series.values
        ax.plot(x_vals, m_vals, label=name,
                color=color, marker=marker, linestyle=ls, linewidth=2)
        ax.fill_between(x_vals, m_vals - s_vals, m_vals + s_vals,
                        color=color, alpha=0.15)
    ax.set_xlabel(xlabel, fontweight="bold")
    ax.set_ylabel("Test ROC-AUC (mean across seeds; shaded = +/- 1 std)", fontweight="bold")
    ax.set_title(title, fontweight="bold", pad=10)
    ax.legend(frameon=True, fancybox=True, shadow=True)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


_plot_comparison(
    gauss_agg, "noise_level",
    "Gaussian Noise Level (fraction of training std, no mean shift)",
    "Robustness on Test Set: Additive Gaussian Noise on Continuous Features\n"
    "(all 4 models, averaged across seeds)",
    os.path.join(comparison_dir, "gaussian_noise_comparison_test.png"))

_plot_comparison(
    dummy_agg, "flip_rate",
    "Corruption Fraction (proportion of dummy cells randomised)",
    "Robustness on Test Set: Random Corruption Noise on Binary (Dummy) Features\n"
    "(all 4 models, averaged across seeds)",
    os.path.join(comparison_dir, "dummy_flip_comparison_test.png"))

print(f"\nAll-models comparison plots and CSVs saved to: {comparison_dir}")

Running noise sweeps for all 4 models across all seeds...
  Seed 42 done (features: multi=71, single=122, all=278, forward=139)
  Seed 43 done (features: multi=80, single=117, all=278, forward=139)
  Seed 44 done (features: multi=82, single=127, all=278, forward=139)
  Seed 45 done (features: multi=76, single=116, all=278, forward=139)
  Seed 46 done (features: multi=79, single=113, all=278, forward=139)
  Seed 47 done (features: multi=83, single=118, all=278, forward=139)
  Seed 48 done (features: multi=81, single=125, all=278, forward=139)
  Seed 49 done (features: multi=81, single=111, all=278, forward=139)
  Seed 50 done (features: multi=78, single=114, all=278, forward=139)
  Seed 51 done (features: multi=71, single=115, all=278, forward=139)
  Seed 52 done (features: multi=74, single=121, all=278, forward=139)
  Seed 53 done (features: multi=68, single=102, all=278, forward=139)
  Seed 54 done (features: multi=78, single=126, all=278, forward=139)
  Seed 55 done (features: multi=

## Paired Wilcoxon Signed-Rank Tests Across Seeds

For each pair of models (MOGA, SOGA, all-features, forward-stepwise) compute the **paired Wilcoxon signed-rank test** on per-seed test AUC. The pairing is by seed: at each noise level all four models are evaluated on identical data (same train/test split, same noisy `X_test` realisation), so the per-seed AUCs are matched samples ﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿ exactly what Wilcoxon assumes.

Reported quantities per pair:
- `mean_a`, `mean_b` ﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿ mean test AUC across seeds.
- `mean_diff`, `median_diff` ﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿ paired difference (a ﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿ b) summary.
- `wilcoxon_statistic`, `p_value` ﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿ two-sided test (no normality assumption).
- `significant_0.05` ﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿ convenience boolean for the conventional 0.05 cutoff.

Three tables are produced:
1. **Clean test set** (noise_level = 0.0) ﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿ the headline comparison.
2. **Across all Gaussian noise levels** ﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿ to see where significance appears/disappears under noise.
3. **Across all dummy corruption rates** ﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿ same, for the binary-feature noise sweep.

This depends on `gauss_df` and `dummy_df` produced by the *all-models comparison* cell above; run that cell first.

In [18]:
# Paired Wilcoxon signed-rank tests across seeds (depends on gauss_df, dummy_df)
from scipy.stats import wilcoxon

# Pairs to compare (model_a, model_b) using the auc_<key> columns in gauss_df / dummy_df
wilcoxon_pairs: list[tuple[str, str]] = [
    ("multi",   "single"),
    ("multi",   "all"),
    ("multi",   "forward"),
    ("single",  "all"),
    ("single",  "forward"),
    ("all",     "forward"),
]

wilcoxon_label_map: dict[str, str] = {
    "multi":   "MOGA (SiCo)",
    "single":  "SOGA (AUC)",
    "all":     "All features",
    "forward": "Forward stepwise",
}


def _paired_wilcoxon_table(per_seed_df: pandas.DataFrame, level_col: str,
                           level_value: float) -> pandas.DataFrame:
    """Run paired Wilcoxon on every model pair at a fixed noise level.

    Returns one row per pair with means, paired-diff summary, Wilcoxon
    statistic, two-sided p-value, and a 0.05 significance flag.
    """
    sub: pandas.DataFrame = per_seed_df[
        numpy.isclose(per_seed_df[level_col], level_value)
    ]
    n_seeds: int = int(sub["seed"].nunique())
    rows: list[dict] = []
    for a, b in wilcoxon_pairs:
        x: numpy.ndarray = sub[f"auc_{a}"].to_numpy()
        y: numpy.ndarray = sub[f"auc_{b}"].to_numpy()
        diff: numpy.ndarray = x - y

        if numpy.allclose(diff, 0.0):
            stat: float = float("nan")
            p_val: float = 1.0
        else:
            stat_raw, p_raw = wilcoxon(
                x, y, zero_method="wilcox", alternative="two-sided")
            stat = float(stat_raw)
            p_val = float(p_raw)

        rows.append({
            "model_a":            wilcoxon_label_map[a],
            "model_b":            wilcoxon_label_map[b],
            "n_seeds":            n_seeds,
            "mean_a":             float(numpy.mean(x)),
            "mean_b":             float(numpy.mean(y)),
            "mean_diff":          float(numpy.mean(diff)),
            "median_diff":        float(numpy.median(diff)),
            "wilcoxon_statistic": stat,
            "p_value":            p_val,
            "significant_0.05":   bool(p_val < 0.05),
        })
    return pandas.DataFrame(rows)


# --- Headline: clean test set (Gaussian sweep at noise_level = 0.0) ---
clean_wilcoxon: pandas.DataFrame = _paired_wilcoxon_table(gauss_df, "noise_level", 0.0)

print("=== Paired Wilcoxon Signed-Rank Test: CLEAN Test AUC ===")
print(f"  n_seeds = {clean_wilcoxon['n_seeds'].iloc[0]} | alternative = two-sided\n")
print(clean_wilcoxon[[
    "model_a", "model_b", "mean_a", "mean_b",
    "mean_diff", "median_diff", "p_value", "significant_0.05",
]].to_string(index=False))

# --- Per-level: Wilcoxon at every Gaussian noise level ---
gauss_levels: list[float] = sorted(gauss_df["noise_level"].unique())
gauss_chunks: list[pandas.DataFrame] = []
for lvl in gauss_levels:
    chunk: pandas.DataFrame = _paired_wilcoxon_table(gauss_df, "noise_level", float(lvl))
    chunk["noise_level"] = float(lvl)
    gauss_chunks.append(chunk)
gauss_wilcoxon_df: pandas.DataFrame = pandas.concat(gauss_chunks, ignore_index=True)

# --- Per-level: Wilcoxon at every dummy corruption rate ---
flip_rates: list[float] = sorted(dummy_df["flip_rate"].unique())
flip_chunks: list[pandas.DataFrame] = []
for r in flip_rates:
    chunk = _paired_wilcoxon_table(dummy_df, "flip_rate", float(r))
    chunk["flip_rate"] = float(r)
    flip_chunks.append(chunk)
dummy_wilcoxon_df: pandas.DataFrame = pandas.concat(flip_chunks, ignore_index=True)

# --- Save outputs ---
wilcoxon_dir: str = os.path.join(RESULT_PATH_EVAL, "wilcoxon")
_ensure_directory(wilcoxon_dir)
clean_wilcoxon.to_csv(
    os.path.join(wilcoxon_dir, "wilcoxon_clean.csv"), index=False)
gauss_wilcoxon_df.to_csv(
    os.path.join(wilcoxon_dir, "wilcoxon_gaussian_per_level.csv"), index=False)
dummy_wilcoxon_df.to_csv(
    os.path.join(wilcoxon_dir, "wilcoxon_dummy_per_level.csv"), index=False)

# --- Quick summary: where MOGA significantly beats each baseline under noise ---
print("\n=== Where MOGA significantly differs from each baseline (Gaussian sweep) ===")
moga_rows: pandas.DataFrame = gauss_wilcoxon_df[
    gauss_wilcoxon_df["model_a"] == wilcoxon_label_map["multi"]
]
for opponent in [wilcoxon_label_map["single"], wilcoxon_label_map["all"], wilcoxon_label_map["forward"]]:
    sub = moga_rows[moga_rows["model_b"] == opponent]
    sig_levels = sub[sub["significant_0.05"]]["noise_level"].tolist()
    print(f"  MOGA vs {opponent:<18}: significant at {len(sig_levels)}/{len(sub)} noise levels "
          f"(p<0.05) -> {sig_levels}")

print(f"\nWilcoxon results saved to: {wilcoxon_dir}")

=== Paired Wilcoxon Signed-Rank Test: CLEAN Test AUC ===
  n_seeds = 20 | alternative = two-sided

     model_a          model_b   mean_a   mean_b  mean_diff  median_diff  p_value  significant_0.05
 MOGA (SiCo)       SOGA (AUC) 0.820270 0.793309   0.026962     0.030950 0.000395              True
 MOGA (SiCo)     All features 0.820270 0.810593   0.009677     0.008173 0.021484              True
 MOGA (SiCo) Forward stepwise 0.820270 0.810157   0.010113     0.008391 0.067259             False
  SOGA (AUC)     All features 0.793309 0.810593  -0.017284    -0.020161 0.000082              True
  SOGA (AUC) Forward stepwise 0.793309 0.810157  -0.016848    -0.013949 0.001209              True
All features Forward stepwise 0.810593 0.810157   0.000436    -0.000763 0.925634             False

=== Where MOGA significantly differs from each baseline (Gaussian sweep) ===
  MOGA vs SOGA (AUC)        : significant at 11/11 noise levels (p<0.05) -> [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0

## Feature Count Comparison Across Seeds

For each method (MOGA, SOGA, all-features, forward-stepwise) plot the number of selected features per seed.

- MOGA and SOGA vary with the random seed (different GA inits / CV shuffles produce different Pareto fronts and HoFs).
- "All features" is by construction constant at `n_features`.
- Forward stepwise is constant at the matched-budget `moga_target_features` value (option-b methodology choice).

The plot is a boxplot (quartile summary) with individual seeds overlaid as jittered dots, so both the distribution shape and the per-seed values are visible. Depends on `best_multi_by_seed`, `best_single_by_seed` (from the stability cell) and `baseline_fwd_df` (from the forward-stepwise baseline cell).

In [19]:
# Feature count comparison across seeds
import seaborn as sns

feature_count_rows: list[dict] = []
for s in seeds:
    feature_count_rows.append({
        "method": "MOGA (SiCo)",
        "seed": s,
        "n_features": int(sum(best_multi_by_seed[s])),
    })
    feature_count_rows.append({
        "method": "SOGA (AUC)",
        "seed": s,
        "n_features": int(sum(best_single_by_seed[s])),
    })
    feature_count_rows.append({
        "method": "All features",
        "seed": s,
        "n_features": int(len(feature_names)),
    })

# Forward stepwise: pull directly from the per-seed baseline table.
for _, _row in baseline_fwd_df.iterrows():
    feature_count_rows.append({
        "method": "Forward stepwise",
        "seed": int(_row["seed"]),
        "n_features": int(_row["n_features"]),
    })

feature_count_df: pandas.DataFrame = pandas.DataFrame(feature_count_rows)

print("=== Feature Count Comparison Across Seeds ===\n")
summary_df: pandas.DataFrame = (
    feature_count_df.groupby("method")["n_features"]
    .agg(["min", "median", "mean", "max", "std"])
    .round(2)
)
print(summary_df.to_string())

# --- Plot: boxplot + stripplot overlay ---
method_order: list[str] = ["MOGA (SiCo)", "SOGA (AUC)", "Forward stepwise", "All features"]
palette: dict[str, str] = {
    "MOGA (SiCo)":      "tab:blue",
    "SOGA (AUC)":       "tab:orange",
    "Forward stepwise": "tab:red",
    "All features":     "tab:green",
}

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(
    data=feature_count_df,
    x="method", y="n_features",
    order=method_order, hue="method", palette=palette,
    legend=False, width=0.5, ax=ax,
)
sns.stripplot(
    data=feature_count_df,
    x="method", y="n_features",
    order=method_order,
    color="black", size=4, alpha=0.6, jitter=0.15, ax=ax,
)

# Annotate each box with the median value
for i, method in enumerate(method_order):
    med: float = feature_count_df.loc[feature_count_df["method"] == method, "n_features"].median()
    ax.annotate(f"median={int(med)}",
                xy=(i, med), xytext=(0, 14),
                textcoords="offset points", ha="center",
                fontsize=9, color="black",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8))

ax.set_xlabel("Method", fontweight="bold")
ax.set_ylabel(f"Number of selected features (out of {len(feature_names)})", fontweight="bold")
ax.set_title(f"Feature Count per Method Across {len(seeds)} Seeds\n"
             f"(box = IQR with median line, dots = individual seeds)",
             fontweight="bold", pad=10)
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()

feature_count_dir: str = os.path.join(RESULT_PATH_EVAL, "feature_counts")
_ensure_directory(feature_count_dir)
fig.savefig(os.path.join(feature_count_dir, "feature_count_comparison.png"), dpi=150)
plt.close(fig)

feature_count_df.to_csv(
    os.path.join(feature_count_dir, "feature_count_per_seed.csv"), index=False)
summary_df.to_csv(
    os.path.join(feature_count_dir, "feature_count_summary.csv"))

print(f"\nFeature count plot and CSVs saved to: {feature_count_dir}")

=== Feature Count Comparison Across Seeds ===

                  min  median    mean  max   std
method                                          
All features      278   278.0  278.00  278  0.00
Forward stepwise  139   139.0  139.00  139  0.00
MOGA (SiCo)        68    78.5   77.85   85  4.74
SOGA (AUC)        102   116.5  116.80  127  6.55

Feature count plot and CSVs saved to: 2026-05-15_15-55-20\evaluation\feature_counts
